# Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from collections import defaultdict, Counter
import pandas as pd
import os

# Functions

In [ ]:
def is_tie(v: str) -> bool:
    return "tie" in v.lower() or v == "C"


def read_all_file(folder_path):
    return os.listdir(folder_path)


def verdicts_agree(v1: str, v2: str) -> bool:
    """True when both verdicts point to the same winner (ties included)."""
    return v1 == v2


# ---------------------------------------------------------------------------
# Parquet reader  (wraps pyarrow / fastparquet with a helpful error)
# ---------------------------------------------------------------------------

def read_parquet(path: str) -> pd.DataFrame:
    """
    Read a Parquet file using pyarrow (preferred) or fastparquet.
    Raises a clear ImportError message if neither is installed.
    """
    try:
        return pd.read_parquet(path, engine="pyarrow")
    except ImportError:
        pass
    try:
        return pd.read_parquet(path, engine="fastparquet")
    except ImportError:
        raise ImportError(
            "Neither 'pyarrow' nor 'fastparquet' is installed.\n"
            "Install one of them with:  pip install pyarrow"
        )


# ---------------------------------------------------------------------------
# Data loading & normalisation
# ---------------------------------------------------------------------------

def load_ai_judge(folder_path: str) -> pd.DataFrame:
    """
    Load the AI judge CSV and return one row per question with columns:
      question_id, model_a, model_b, verdict, category
    """
    df = pd.read_csv(folder_path)

    required = {"question_id", "model_a", "model_b", "verdict", "judge_name"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"AI CSV is missing columns: {missing}")

    print(f"[AI judge]  Loaded {len(df)} rows from '{folder_path}'")
    return df


def load_human_judge(human_path: str) -> pd.DataFrame:
    """
    Load the human judge Parquet and return a DataFrame with columns:
      question_id, model_a, model_b, verdict, judge_name (or judge_email)
    """
    df = pd.read_csv(human_path)

    required = {"question_id", "model_a", "model_b", "verdict", 'judge_name'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Human Parquet is missing columns: {missing}")

    print(f"[Human judge] Loaded {len(df)} rows from '{human_path}'")
    return df


# ---------------------------------------------------------------------------
# Agreement computation
# ---------------------------------------------------------------------------

def build_vote_index(df: pd.DataFrame):
    """
    Build a dict: (question_id, judge_name) -> list[verdict]
    """
    index = defaultdict(list)
    for _, row in df.iterrows():
        key = (row["question_id"])
        index[key].append(row["verdict"])
    return index

def compute_human_agreement_majority(
    ai_index: dict,
    human_index: dict,
    exclude_tie: bool = False,
) -> tuple:
    """
    Compute agreement between one AI judgment and multiple human judgments
    on matching question/model-pair keys.

    Parameters
    ----------
    ai_index    : {key: [ai_verdict]}   (always 1 verdict per key)
    human_index : {key: [human_verdict, ...]}  (1 or more per key)
    exclude_tie : if True, skip comparisons where EITHER vote is a tie

    Returns
    -------
    (n_agree, n_total)
    """
    n_agree = 0
    n_total = 0

    common_keys = set(ai_index.keys()) & set(human_index.keys())

    for key in common_keys:
        human_votes = human_index[key]

        if exclude_tie:
            for v in human_votes:
                if is_tie(v):
                    human_votes.remove(v)
        if len(human_votes) == 0:
            continue

        counter = Counter(human_votes)
        max_freq = max(counter.values())

        most_common_values = [k for k, v in counter.items() if v == max_freq]
        len_values = len(most_common_values)
        len_votes = len(human_votes)

        n_agree += max_freq/len(human_votes)
        n_total += 1

    return n_agree, n_total


def compute_agreement_majority(
    ai_index: dict,
    human_index: dict,
    exclude_tie: bool = False,
) -> tuple:
    """
    Compute agreement between one AI judgment and multiple human judgments
    on matching question/model-pair keys.

    Parameters
    ----------
    ai_index    : {key: [ai_verdict]}   (always 1 verdict per key)
    human_index : {key: [human_verdict, ...]}  (1 or more per key)
    exclude_tie : if True, skip comparisons where EITHER vote is a tie

    Returns
    -------
    (n_agree, n_total)
    """
    n_agree = 0
    n_total = 0

    common_keys = set(ai_index.keys()) & set(human_index.keys())

    for key in common_keys:
        ai_vote = ai_index[key][0]  # single AI judgment
        human_votes = human_index[key]

        if exclude_tie:
            if is_tie(ai_vote):
                continue
            for v in human_votes:
                if is_tie(v):
                    human_votes.remove(v)
        if len(human_votes) == 0:
            continue

        counter = Counter(human_votes)
        max_freq = max(counter.values())

        most_common_values = [k for k, v in counter.items() if v == max_freq]
        len_values = len(most_common_values)
        len_votes = len(human_votes)

        if ai_vote in most_common_values:
            n_agree += max_freq/len(human_votes)
        n_total += 1

    return n_agree, n_total


def compute_agreement(
    ai_index: dict,
    human_index: dict,
    exclude_tie: bool = False,
) -> tuple:
    """
    Compute agreement between one AI judgment and multiple human judgments
    on matching question/model-pair keys.

    Parameters
    ----------
    ai_index    : {key: [ai_verdict]}   (always 1 verdict per key)
    human_index : {key: [human_verdict, ...]}  (1 or more per key)
    exclude_tie : if True, skip comparisons where EITHER vote is a tie

    Returns
    -------
    (n_agree, n_total)
    """
    n_agree = 0
    n_total = 0

    common_keys = set(ai_index.keys()) & set(human_index.keys())

    for key in common_keys:
        ai_vote = ai_index[key][0]  # single AI judgment
        human_votes = human_index[key]

        for h_vote in human_votes:
            if exclude_tie:
                if is_tie(ai_vote) or is_tie(h_vote):
                    continue
            n_total += 1
            if verdicts_agree(ai_vote, h_vote):
                n_agree += 1

    return n_agree, n_total


def compute_human_human_agreement(
    ai_index: dict,
    human_index: dict,
    exclude_tie: bool = False,
) -> tuple:
    """
    Compute human-human agreement (inter-rater reliability).
    For each question, compare every pair of human votes.

    Returns
    -------
    (n_agree, n_total)
    """
    n_agree = 0
    n_total = 0
    common_keys = set(ai_index.keys()) & set(human_index.keys())

    for key in common_keys:
        votes = human_index[key]
        if len(votes) < 2:
            continue
        for i in range(len(votes) - 1):
            for j in range(i + 1, len(votes)):
                v1, v2 = votes[i], votes[j]
                if exclude_tie:
                    if is_tie(v1) or is_tie(v2):
                        continue
                n_total += 1
                if verdicts_agree(v1, v2):
                    n_agree += 1

    return n_agree, n_total


# ---------------------------------------------------------------------------
# Category-wise breakdown
# ---------------------------------------------------------------------------

def compute_category_agreement(
    ai_df: pd.DataFrame,
    human_index: dict,
    exclude_tie: bool = False,
) -> dict:
    """
    Compute AI-vs-Human agreement broken down by question category.
    Requires the AI dataframe to have a 'category' column.
    Returns {category: (n_agree, n_total, agreement_ratio)}
    """
    results = {}
    if "category" not in ai_df.columns:
        return results

    for cat, group in ai_df.groupby("category"):
        cat_ai_index = build_vote_index(group)
        n_agree, n_total = compute_agreement(
            cat_ai_index, human_index, exclude_tie=exclude_tie
        )
        if n_total > 0:
            results[cat] = (n_agree, n_total, n_agree / n_total)
        else:
            results[cat] = (0, 0, float("nan"))
    return results


# ---------------------------------------------------------------------------
# Main runner
# ---------------------------------------------------------------------------

def run(ai_csv: str, human_parquet: str) -> None:

    # ---------- Load data ----------
    ai_df = load_ai_judge(ai_csv)
    human_df = load_human_judge(human_parquet)

    ai_index = build_vote_index(ai_df)
    human_index = build_vote_index(human_df)

    common = set(ai_index.keys()) & set(human_index.keys())
    print(f"\nOverlapping question/model-pair keys: {len(common)}")
    if len(common) == 0:
        print(
            "\n[WARNING] No matching question_id + model_a + model_b keys found "
            "between the two files.\n"
            "Check that the question_ids and model names align between CSV and Parquet.\n"
            f"  AI  keys (first 3): {list(ai_index.keys())[:3]}\n"
            f"  Human keys (first 3): {list(human_index.keys())[:3]}"
        )
        return

    # ---------- S1: with tie ----------
    print("\n--- Setup S1: ALL votes (ties counted) ---")
    print(f"  Baseline (random agreement): ~33%")

    m_agree_s1, m_total_s1 = compute_agreement_majority(ai_index, human_index, exclude_tie=False)
    m_ratio_s1 = m_agree_s1 / m_total_s1
    print(f"  AI vs Human (majority)  | total: {m_total_s1:5d} | agree: {m_agree_s1:5f} | ratio: {m_ratio_s1:.3f}")

    n_agree_s1, n_total_s1 = compute_agreement(ai_index, human_index, exclude_tie=False)
    ratio_s1 = n_agree_s1 / n_total_s1 if n_total_s1 > 0 else float("nan")
    print(f"  AI vs Human  | total: {n_total_s1:5d} | agree: {n_agree_s1:5d} | ratio: {ratio_s1:.3f}")

    hh_agree_s1, hh_total_s1 = compute_human_human_agreement(ai_index, human_index, exclude_tie=False)
    hh_ratio_s1 = hh_agree_s1 / hh_total_s1 if hh_total_s1 > 0 else float("nan")
    print(f"  Human vs Human | total: {hh_total_s1:5d} | agree: {hh_agree_s1:5d} | ratio: {hh_ratio_s1:.3f}")

    hhm_agree_s1, hhm_total_s1 = compute_human_agreement_majority(ai_index, human_index, exclude_tie=False)
    hhm_ratio_s1 = hhm_agree_s1 / hhm_total_s1 if hhm_total_s1 > 0 else float("nan")
    print(f"  Human vs Majority | total: {hhm_total_s1:5d} | agree: {hhm_agree_s1:5f} | ratio: {hhm_ratio_s1:.3f}")



    # ---------- S2: without tie ----------
    print("\n--- Setup S2: NON-TIE votes only ---")
    print(f"  Baseline (random agreement): ~50%")

    m_agree_s2, m_total_s2 = compute_agreement_majority(ai_index, human_index, exclude_tie=True)
    m_ratio_s2 = m_agree_s2 / m_total_s2
    print(f"  AI vs Human (majority) | total: {m_total_s2:5d} | agree: {m_agree_s2:5f} | ratio: {m_ratio_s2:.3f}")

    n_agree_s2, n_total_s2 = compute_agreement(ai_index, human_index, exclude_tie=True)
    ratio_s2 = n_agree_s2 / n_total_s2 if n_total_s2 > 0 else float("nan")
    print(f"  AI vs Human  | total: {n_total_s2:5d} | agree: {n_agree_s2:5d} | ratio: {ratio_s2:.3f}")

    hh_agree_s2, hh_total_s2 = compute_human_human_agreement(ai_index, human_index, exclude_tie=True)
    hh_ratio_s2 = hh_agree_s2 / hh_total_s2 if hh_total_s2 > 0 else float("nan")
    print(f"  Human vs Human | total: {hh_total_s2:5d} | agree: {hh_agree_s2:5d} | ratio: {hh_ratio_s2:.3f}")

    hhm_agree_s2, hhm_total_s2 = compute_human_agreement_majority(ai_index, human_index, exclude_tie=True)
    hhm_ratio_s2 = hhm_agree_s2 / hhm_total_s2 if hhm_total_s2 > 0 else float("nan")
    print(f"  Human vs Majority | total: {hhm_total_s2:5d} | agree: {hhm_agree_s2:5f} | ratio: {hhm_ratio_s2:.3f}")

    # ---------- Category breakdown ----------
    print("\n--- Category-wise AI vs Human agreement (S2, no tie) ---")
    cat_results = compute_category_agreement(ai_df, human_index, exclude_tie=True)
    if cat_results:
        print(f"  {'Category':<15}  {'agree':>6}  {'total':>6}  {'ratio':>7}")
        print("  " + "-" * 40)
        for cat in sorted(cat_results):
            n_ag, n_tot, ratio = cat_results[cat]
            print(f"  {cat:<15}  {n_ag:6d}  {n_tot:6d}  {ratio:7.3f}")
    else:
        print("  (no category column found in AI CSV)")

    # ---------- Summary ----------
    print("\n" + "=" * 65)
    print("  SUMMARY")
    print("=" * 65)
    print(f"  AI judge : {ai_df['judge_name'].iloc[0] if 'judge_name' in ai_df.columns else 'CSV'}")
    print(f"  AI model : {ai_df['model_a'].iloc[0]}  vs  {ai_df['model_b'].iloc[0]}")
    print(f"  AI rows  : {len(ai_index.keys())}")
    print(f"  Human rows: {len(human_index.keys())}")
    print()
    print(f"  S1 (with tie)    AI-vs-Human : {ratio_s1:.1%}  |  Human-Human : {hh_ratio_s1:.1%}")
    print(f"  S2 (no tie)      AI-vs-Human : {ratio_s2:.1%}  |  Human-Human : {hh_ratio_s2:.1%}")
    print()
    print("  Reference (from MT-Bench paper, Table 5):")
    print("    GPT-4 vs Human (S2, no tie) : ~85%")
    print("    Human vs Human  (S2, no tie): ~81%")
    print("=" * 65)

    results = {}
    results['Ties'] = {'AI_vs_Human': ratio_s1, 'Human_vs_Human': hh_ratio_s1, 'Upper_limit':hhm_ratio_s1}
    results['No_ties'] = {'AI_vs_Human': ratio_s2, 'Human_vs_Human': hh_ratio_s2, 'Upper_limit':hhm_ratio_s2}
    results['Category'] = cat_results
    return results


In [ ]:
paths = {}
models = ['openai', 'moonshotai', 'llama', 'qwen', 'tinyllama']
for model in models:
    folder_path_1 = f'drive/MyDrive/ViJudge/data/data_for_agreement/LLM_judge/{model}_pair.csv'
    folder_path_2 = f'drive/MyDrive/ViJudge/data/data_for_agreement/LLM_judge/{model}_pair_from_single.csv'
    human_path = f"drive/MyDrive/ViJudge/data/data_for_agreement/human_judge/ViJudge.csv"
    paths[model] = (folder_path_1, folder_path_2, human_path)

results = pd.DataFrame()
for model in models:
    folder_path_1, folder_path_2, human_path = paths[model]
    result= run(folder_path_1, human_path)
    result['model'] = model
    result['type'] = 'pair'
    results = pd.concat([results, pd.DataFrame([result])])
    result= run(folder_path_2, human_path)
    result['model'] = model
    result['type'] = 'single'
    results = pd.concat([results, pd.DataFrame([result])])

base_path = 'drive/MyDrive/ViJudge/data/data_for_agreement/'
results.to_csv(f'{base_path}results.csv', index=False)

